# NB-08: T-1 総合勝率予測（完全版）→ T-2 連対率・T-3 複勝率導出

## モデル概要

本 notebook は **アプリケーションの核**となる最終予測モデル。
前段モデル（T-4/T-5/T-6/T-8/T-9）の OOF 出力をすべて特徴量として取り込んだ
**ステッキングアーキテクチャ**を採用する。

```
Stage 1: T-6 (脚質分類) ──────────────────────────────────────────────┐
Stage 2: T-8 (ペース) ──────────────────────────────────────────────── │
         T-4 (上り3F) ──────────────────────────────────────────────── ├─→ T-1 (勝率) → softmax → p_win
         T-5 (位置取り) ─────────────────────────────────────────────── │    → Harville → p_place, p_show
         T-9 (走破タイム) ───────────────────────────────────────────── ┘
         [基本特徴量 + JT 統計 + 種牡馬統計]
```

## 評価指標
- **モデル精度**: AUC, Log Loss
- **馬券収支シミュレーション**: 単勝・複勝 期待回収率 (ROI)


In [ ]:
import sys
sys.path.insert(0, "/home/jovyan/work/keiba-vpn")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss

from src.pipeline.models.notebook_utils import (
    load_master, load_raw_results, encode_cats, feature_set,
    train_lgb, oof_predict, eval_classification, save_oof, load_oof,
    DEFAULT_PARAMS_BINARY, SURFACE_CATS, OOF_DIR,
    softmax_normalize, harville_place, harville_show, apply_harville_per_race,
)
print("imports OK")


## 1. データ読み込み + 前段モデル OOF の追加

In [ ]:
df = load_master()

# ── OOF 予測を順次マージ ──────────────────────────────────────────────
oof_files = {
    "t4": ("t4_oof.parquet",  "t4_oof"),
    "t5": ("t5_oof.parquet",  "t5_oof"),
    "t6": ("t6_oof.parquet",  None),       # 複数列
    "t8": ("t8_oof.parquet",  None),       # 複数列 + race_id キー
    "t9": ("t9_oof.parquet",  "t9_oof"),
}

for name, (fname, col) in oof_files.items():
    p = OOF_DIR / fname
    if not p.exists():
        print(f"WARN: {fname} not found, skip")
        continue
    oof_df = pd.read_parquet(p)
    # マージキー確認
    merge_keys = ["race_id", "horse_number"] if "horse_number" in oof_df.columns else ["race_id"]
    if col:
        df = df.merge(oof_df[merge_keys + [col]], on=merge_keys, how="left")
    else:
        oof_cols = [c for c in oof_df.columns if c.startswith(f"{name}_")]
        df = df.merge(oof_df[merge_keys + oof_cols], on=merge_keys, how="left")
    print(f"  {name} OOF merged: {', '.join(oof_cols if not col else [col])}")

print(f"\nmerge 後 shape: {df.shape}")
df["surface_cat"] = df["surface_cat"].astype(str)  # Categorical → str


## 2. T-6/T-8 の race_id 単位 OOF を horse 行に展開

In [ ]:
# T-8 は race_id キーなので horse 行にブロードキャスト
t8_oof_path = OOF_DIR / "t8_oof.parquet"
if t8_oof_path.exists():
    t8 = pd.read_parquet(t8_oof_path)
    t8_cols = [c for c in t8.columns if c.startswith("t8_prob_")]
    if t8_cols and "t8_prob_H" not in df.columns:
        # race_id 単位のマップ
        t8_race = t8.groupby("race_id")[t8_cols].first().reset_index()
        df = df.merge(t8_race, on="race_id", how="left")
        print(f"T-8 race-level OOF merged: {t8_cols}")

print("OOF 列 coverage:")
oof_cols = [c for c in df.columns if c.startswith(("t4_","t5_","t6_","t8_","t9_"))]
for c in oof_cols:
    print(f"  {c}: {df[c].notna().mean():.1%}")


## 3. 特徴量定義（総合版）

In [ ]:
# T-1 用特徴量: 全基本特徴量 + OOF スタック + NB-02 種牡馬統計 (利用可能なら)
FEAT_T1 = feature_set(df, extra=oof_cols + [
    "speed_max", "speed_avg", "speed_distance",
    "speed_recent_1", "speed_recent_2", "speed_recent_3",
    # 種牡馬統計 (nb-02 実行後に利用可能)
    "sire_prior_win_rate", "sire_prior_win_rate_surface", "sire_prior_win_rate_dist",
    "dam_sire_prior_win_rate", "dam_sire_prior_win_rate_surface",
])
print(f"T-1 特徴量数: {len(FEAT_T1)}")

# NaN 率確認
nan_rates = df[FEAT_T1].isna().mean().sort_values(ascending=False).head(10)
print("\nNaN 率 Top 10:")
print(nan_rates.to_string())

CAT_USE = [c for c in ["venue","surface","direction","grade","track_condition","weather","sex"] if c in df.columns]
df = encode_cats(df, CAT_USE)


## 4. 馬場別モデル学習（完全版）

In [ ]:
params_t1 = {
    **DEFAULT_PARAMS_BINARY,
    "num_leaves": 127,
    "min_data_in_leaf": 50,
    "learning_rate": 0.03,
}

models_t1: dict = {}
oof_all_t1 = pd.DataFrame()

for sc in SURFACE_CATS:
    df_sc = df[df["surface_cat"] == sc].copy()
    df_tr = df_sc[df_sc["split"] == "train"]
    df_vl = df_sc[df_sc["split"] == "valid"]
    print(f"\n=== {sc} ===  train={len(df_tr):,}  valid={len(df_vl):,}")
    if len(df_tr) < 100: print("  スキップ"); continue

    model = train_lgb(df_tr, df_vl, FEAT_T1, "is_winner", params_t1, cat_features=CAT_USE)
    models_t1[sc] = model

    # valid 評価
    pred_vl = model.predict(df_vl[FEAT_T1])
    y_vl = df_vl["is_winner"].values
    auc = roc_auc_score(y_vl, pred_vl)
    ll  = log_loss(y_vl, pred_vl)
    print(f"  Valid: AUC={auc:.4f}  LogLoss={ll:.4f}")

    # OOF
    oof = oof_predict(df_tr, FEAT_T1, "is_winner", params_t1, cat_features=CAT_USE)
    oof_df = df_tr[["race_id","horse_number","surface_cat","is_winner","is_top3","split"]].copy()
    oof_df["t1_oof"] = oof.values
    oof_all_t1 = pd.concat([oof_all_t1, oof_df], ignore_index=True)

print("\nモデル学習完了:", list(models_t1.keys()))


## 5. テストセット評価

In [ ]:
for sc, model in models_t1.items():
    df_te = df[(df["surface_cat"] == sc) & (df["split"] == "test")]
    if df_te.empty: continue
    pred_te = model.predict(df_te[FEAT_T1])
    y_te = df_te["is_winner"].values
    auc = roc_auc_score(y_te, pred_te)
    ll  = log_loss(y_te, pred_te)
    print(f"[{sc}] Test: AUC={auc:.4f}  LogLoss={ll:.4f}")

    # 上位N頭の的中率
    df_te_copy = df_te.copy()
    df_te_copy["pred"] = pred_te
    top_k = df_te_copy.groupby("race_id").apply(
        lambda g: g.nlargest(3, "pred")["is_winner"].sum() > 0
    ).mean()
    print(f"  Top-3 hit rate (1レース中に1着馬がTop3内): {top_k:.1%}")


## 6. Harville 式による連対率・複勝率導出

In [ ]:
# テストセットで Harville 式適用例
sc = [s for s in models_t1 if s == "turf" or s == SURFACE_CATS[0]][0]
df_te = df[(df["surface_cat"] == sc) & (df["split"] == "test")].copy()
df_te["win_prob_raw"] = models_t1[sc].predict(df_te[FEAT_T1])

# レースごとに softmax + Harville
df_te = apply_harville_per_race(df_te, "win_prob_raw")

# 精度確認
print(f"[{sc}] Test set での連対率・複勝率の精度確認")
# is_top3 ≈ show_prob の一致率（相関）
from scipy.stats import spearmanr
valid = df_te["is_top3"].notna() & df_te["show_prob"].notna()
sr, _ = spearmanr(df_te.loc[valid, "is_top3"], df_te.loc[valid, "show_prob"])
print(f"  is_top3 vs show_prob Spearman: {sr:.4f}")
print(df_te[["race_id","horse_number","win_prob_raw","place_prob","show_prob"]].head(18).to_string())


## 7. 馬券収支シミュレーション（単勝）

In [ ]:
# 単純シミュレーション: 各レースで最高 win_prob の馬に単勝賭けた場合の的中率
print("=== 馬券収支シミュレーション（単勝, テストセット） ===")
for sc, model in models_t1.items():
    df_te = df[(df["surface_cat"] == sc) & (df["split"] == "test")].copy()
    if df_te.empty: continue
    df_te["win_prob"] = model.predict(df_te[FEAT_T1])

    # レースごとに最大確率の馬を選択
    best = df_te.groupby("race_id").apply(lambda g: g.loc[g["win_prob"].idxmax()])
    hit_rate = best["is_winner"].mean()
    n_races  = len(best)
    print(f"[{sc}] 対象レース数: {n_races:,}  単勝的中率: {hit_rate:.1%}")
    # 理論上の平均 1/field_size が基準
    avg_field = df_te.groupby("race_id")["field_size"].first().mean()
    baseline  = 1 / avg_field
    print(f"  ランダム基準: {baseline:.1%}  Lift: {hit_rate/baseline:.2f}x")


## 8. OOF 保存 & 特徴量重要度

In [ ]:
if not oof_all_t1.empty:
    oof_all_t1.to_parquet(OOF_DIR / "t1_oof.parquet", index=False)
    print(f"T-1 OOF saved: shape={oof_all_t1.shape}")

best_sc = max(models_t1, key=lambda s: models_t1[s].num_trees()) if models_t1 else None
if best_sc:
    imp = pd.Series(models_t1[best_sc].feature_importance(importance_type="gain"), index=FEAT_T1)
    imp_top = imp.sort_values(ascending=False).head(25)
    fig, ax = plt.subplots(figsize=(10, 8))
    imp_top.plot.barh(ax=ax)
    ax.set_title(f"T-1 (Win Probability) Feature Importance [{best_sc}]")
    plt.tight_layout()
    plt.savefig(OOF_DIR / "t1_feature_importance.png", dpi=80)
    plt.show()
    print("\nTop 15 features:")
    print(imp_top.head(15).to_dict())


## 9. まとめ

| モデル | ターゲット | 評価 |
|---|---|---|
| T-6 (nb-03) | 脚質分類 (逃/先/差/追) | Accuracy, F1-macro |
| T-8 (nb-04) | ペースカテゴリ (H/M/S) | F1-macro |
| T-4 (nb-05) | 上り3Fタイム | MAE (秒), Spearman |
| T-5 (nb-06) | 位置取り (正規化着順) | Spearman, MAE |
| T-9 (nb-07) | 走破タイム | MAE (秒), RMSE |
| **T-1 (本 NB)** | **勝率 → 連対率・複勝率** | **AUC, ROI, 的中率** |

### 本番化の手順
1. 各モデルを `mlflow` に登録（`src/pipeline/models/` 参照）
2. 推論 API で Stage 1 → Stage 2 → Stage 3 の順に実行
3. Harville 式で T-2/T-3 を導出
4. `data/local/modeling/oof/` の OOF を次期学習に使用

### 改善余地
- 種牡馬統計 (nb-02 実行後に自動的に特徴量追加) → AUC +0.01 程度の改善見込み
- Purged GroupKFold のギャップ設定（時系列リークをさらに厳密に制御）
- アンサンブル: XGBoost, CatBoost との Stacking
